In [ ]:
import torch
import torch.nn.functional as F
from importlib import reload
import datasets
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json, os
from itertools import combinations
from analysis_helpers import analyze_routing_entropy, pairwise_mean_jsd
from get_routing_weights import DATA_FOLDER

In [ ]:
## CONFIGS TO CHANGE
MODEL_NAME = "gpt"
display_name = "GPT-OSS 20B"
base_model_file = f"total_actual_weight_flores_{MODEL_NAME}_9langs.json"
middle_layer_boundaries = [4, 17]

filename_dict = {
    "kir_Cyrl": "total_actual_weight_flores_gpt_gpt_kir__L4-17_nofreeze_200k_9langs.json",
    "kan_Knda": "total_actual_weight_flores_gpt_gpt_kan__L4-17_nofreeze_199k_2langs.json",
}
routers_filename_dict = {
    "kan_Knda": "total_actual_weight_flores_gpt_gpt_kan__L4-17_routers_200k_2langs.json",
    "kir_Cyrl": "total_actual_weight_flores_gpt_gpt_kir__L4-17_routers_200k_2langs.json"
}
baseline_filename_dict = {
    "kan_Knda": "total_actual_weight_flores_gpt_gpt_kan_baseline-target_lm_200k_2langs.json",
    "kir_Cyrl": "total_actual_weight_flores_gpt_gpt_kir_baseline-target_lm_200k_2langs.json"
}

# MODEL_NAME = "qwen3_30b"
# display_name = "Qwen3 30B A3B"
# base_model_file = f"total_actual_weight_flores_{MODEL_NAME}_9langs.json"
# middle_layer_boundaries = [7, 34]

# filename_dict = {
#     "kir_Cyrl": "total_actual_weight_flores_qwen3_30b_qwen3_kir__L7-34_nofreeze_200k_9langs.json",
#     "kan_Knda": "total_actual_weight_flores_qwen3_30b_qwen3_kan__L7-34_nofreeze_200k_2langs.json",
#     "tel_Telu": "total_actual_weight_flores_qwen3_30b_qwen3_tel__L7-34_nofreeze_200k_2langs.json"
# }
# routers_filename_dict = {
#     "kan_Knda": "total_actual_weight_flores_qwen3_30b_qwen3_kan__L7-34_routers_200k_2langs.json",
#     "kir_Cyrl": "total_actual_weight_flores_qwen3_30b_qwen3_kir__L7-34_routers_200k_2langs.json",
#     "tel_Telu": "total_actual_weight_flores_qwen3_30b_qwen3_tel__L7-34_routers_200k_2langs.json"
# }
# baseline_filename_dict = {
#     "kan_Knda": "total_actual_weight_flores_qwen3_30b_qwen3_kan_baseline-target_lm_200k_2langs.json",
#     "kir_Cyrl": "total_actual_weight_flores_qwen3_30b_qwen3_kir_baseline-target_lm_200k_2langs.json",
#     "tel_Telu": "total_actual_weight_flores_qwen3_30b_qwen3_tel_baseline-target_lm_200k_2langs.json"
# }

# MODEL_NAME = "granite"
# display_name = "Granite 4-H tiny"
# base_model_file = f"total_actual_weight_flores_{MODEL_NAME}_9langs.json"
# middle_layer_boundaries = [10, 35]

# filename_dict = {
#     "hun_Latn": "total_actual_weight_flores_granite_granite_hun__L10-35_nofreeze_200k_2langs.json",
#     # "hun_Latn": "total_actual_weight_flores_granite_granite_hun__L10-35_routers_199k_2langs.json"
# }
# routers_filename_dict = {
#     "hun_Latn": "total_actual_weight_flores_granite_granite_hun__L10-35_routers_201k_2langs.json"
# }
# baseline_filename_dict = {
#     "hun_Latn": "total_actual_weight_flores_granite_granite_hun_baseline-target_lm_200k_2langs.json"
#     # "hun_Latn": "total_actual_weight_flores_granite_granite_hun__L10-35_routers_201k_2langs.json"
#     # "hun_Latn": "total_actual_weight_flores_granite_granite_hun__L10-35_nofreeze_201k_2langs.json",
# }

# MODEL_NAME = "marco_nano"
# display_name = "Marco Nano"
# base_model_file = f"total_actual_weight_flores_{MODEL_NAME}_9langs.json"
# middle_layer_boundaries = [7, 19]

# filename_dict = {
#     "hun_Latn": "total_actual_weight_flores_marco_nano_marco_hun__L7-19_nofreeze_199k_2langs.json",
#     "sin_Sinh": "total_actual_weight_flores_marco_nano_marco_sin__L7-19_nofreeze_200k_2langs.json"
# }
# routers_filename_dict = {
#     "hun_Latn": "total_actual_weight_flores_marco_nano_marco_hun__L7-19_routers_200k_2langs.json"
#     # "hun_Latn": "total_actual_weight_flores_marco_nano_marco_hun__L7-19_nofreeze_200k_2langs.json",
# }
# baseline_filename_dict = {
#     "hun_Latn": "total_actual_weight_flores_marco_nano_marco_hun_baseline-target_lm_199k_2langs.json",
#     "sin_Sinh": "total_actual_weight_flores_marco_nano_marco_sin_baseline-target_lm_200k_2langs.json"
# }

In [ ]:
def load_reloaded_tensor_data(filename, languages):
    output_filepath = os.path.join(DATA_FOLDER, filename)
    with open(output_filepath, 'r') as f:
        reloaded_data = json.load(f)

    reloaded_tensor_data = {}
    for key, list_of_arrays in reloaded_data.items():
        if key in languages:
            tensor_list = []
            for array in list_of_arrays:
                tensor_list.append(torch.tensor(array))
            reloaded_tensor_data[key] = tensor_list
    return reloaded_tensor_data


base_model_data = load_reloaded_tensor_data(base_model_file, ["eng"]+[k[:3] for k in filename_dict.keys()])
trained_model_data = {}
routers_model_data = {}
baseline_model_data = {}
for key, filename in filename_dict.items():
    trained_model_data[key] = load_reloaded_tensor_data(filename, ["eng", key[:3]])
    if key in routers_filename_dict:
        routers_model_data[key] = load_reloaded_tensor_data(routers_filename_dict[key], ["eng", key[:3]])
    if key in baseline_filename_dict:
        baseline_model_data[key] = load_reloaded_tensor_data(baseline_filename_dict[key], ["eng", key[:3]])

In [ ]:
base_entropy_results = analyze_routing_entropy(base_model_data)
trained_entropy_results = {}
routers_entropy_results = {}
baseline_entropy_results = {}
for key, trained_data in trained_model_data.items():
    trained_entropy_results[key] = analyze_routing_entropy(trained_data)
    if key in routers_model_data:
        routers_entropy_results[key] = analyze_routing_entropy(routers_model_data[key])
    if key in baseline_model_data:
        baseline_entropy_results[key] = analyze_routing_entropy(baseline_model_data[key])

In [ ]:
jsds_per_layer_base = pairwise_mean_jsd(base_model_data, entropy_normalized=True)
jsds_per_layer_trained = {}
jsds_per_layer_routers = {}
jsds_per_layer_baseline = {}
for key, trained_data in trained_model_data.items():
    jsds_per_layer_trained[key] = pairwise_mean_jsd(trained_data, entropy_normalized=True)[key[:3]]
    if key in routers_model_data:
        jsds_per_layer_routers[key] = pairwise_mean_jsd(routers_model_data[key], entropy_normalized=True)[key[:3]]
    if key in baseline_model_data:
        jsds_per_layer_baseline[key] = pairwise_mean_jsd(baseline_model_data[key], entropy_normalized=True)[key[:3]]

In [ ]:
N = int(jsds_per_layer_base[next(iter(jsds_per_layer_base))].size()[0])

# --- Prepare data for plotting ---
# The x-axis will represent the layer index. We can make it 1-indexed for better readability.
layer_indices = np.arange(1, N + 1) # Creates an array from 1 to 48

# colors = dict(zip(jsds_per_layer_base.keys(), plt.cm.get_cmap('tab20')(np.linspace(0, 1, len(jsds_per_layer_base.keys())))))

scale_factor = 2
for lang in jsds_per_layer_trained.keys():
    plt.figure(figsize=(9 * scale_factor, 4 * scale_factor)) # Adjust figure size for better readability of 48 points
    jsd_values = jsds_per_layer_base[lang[:3]].to(torch.float32).cpu().numpy()
    plt.plot(layer_indices, jsd_values, marker='o', linestyle='-', linewidth=2  * scale_factor, markersize=5  * scale_factor, label="base model", color="black")

    jsd_values = jsds_per_layer_trained[lang].to(torch.float32).cpu().numpy()
    plt.plot(layer_indices, jsd_values, marker='o', linestyle='-', linewidth=2  * scale_factor, markersize=5  * scale_factor, label=f"trained", color="green")

    if lang in jsds_per_layer_routers:
        jsd_values = jsds_per_layer_routers[lang].to(torch.float32).cpu().numpy()
        plt.plot(layer_indices, jsd_values, marker='o', linestyle='-', linewidth=2  * scale_factor, markersize=5  * scale_factor, label=f"routers", color="blue")

    if lang in jsds_per_layer_baseline:
        jsd_values = jsds_per_layer_baseline[lang].to(torch.float32).cpu().numpy()
        plt.plot(layer_indices, jsd_values, marker='o', linestyle='-', linewidth=2  * scale_factor, markersize=5  * scale_factor, label=f"baseline", color="red")

    # --- Add labels, title, and grid for clarity ---
    plt.title(
        f"[ {display_name}, {lang} ]",
        y=0.85, fontsize=14 * scale_factor,
        bbox=dict(facecolor='white', edgecolor='none', boxstyle='square,pad=0.4', alpha=1.0)
    )
    plt.suptitle(
        'Routing Divergence from English, per Layer', 
        y=0.9, fontsize=14 * scale_factor, fontweight='bold', 
        bbox=dict(facecolor='white', edgecolor='none', boxstyle='square,pad=0.2', alpha=1.0)
    )
    plt.legend(bbox_to_anchor=(1, 1), loc='upper left', borderaxespad=0., fontsize=9 * scale_factor)
    plt.xlabel('Layer Number', fontsize=12 * scale_factor)
    plt.ylabel('Mean JSD (entropy-normalized)', fontsize=12 * scale_factor)
    xticks = np.append(np.arange(1, N+1, 5), N)
    plt.xticks(xticks) # Set x-axis ticks to show every 5 layers for readability
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.yticks(np.arange(0, 0.6, 0.05))
    plt.xlim(left=0.5, right=N+0.5)
    plt.ylim(bottom=0, top=0.35) # JSD is always non-negative, so set y-axis lower limit to 0

    plt.axvline(x=middle_layer_boundaries[0]+0.5, color='red', linestyle='-', linewidth=1 * scale_factor, alpha=0.7)
    plt.axvline(x=middle_layer_boundaries[1]+1.5, color='red', linestyle='-', linewidth=1 * scale_factor, alpha=0.7)


    # Increase the tick size for x and y axes
    plt.tick_params(axis='both', which='major', labelsize=9 * scale_factor)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    # --- Display the plot ---
    plt.tight_layout() # Adjusts plot to prevent labels from overlapping
    plt.show()

In [ ]:
jsds_per_layer_base["hun"]

In [ ]:
jsds_per_layer_routers["hun_Latn"]